In [44]:
import os
import sys
from sys import platform

sys.path.append('../../')


import tiktoken

import torch
import torch.nn as nn

%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel, generate_text_simple
from llm_from_scratch.CH2.text_data_set import create_dataloader_v1
from llm_from_scratch.CH5.loss import calc_loss_loader, calc_loss_batch

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
def text_to_token_ids(text, tokenizer):
    """
    Args:
        text (str): Sentence or phrase
    Returns:
        (torch.Tensor): Token indices of size (batch_size, num_tokens) where batch_size is 1
    """
    encoded=tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor=torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat=token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

In [5]:
GPT_CONFIG_124M={'vocab_size':50257,
                 'context_length':256,
                 'emb_dim':768,
                 'n_heads':12,
                 'n_layers':12,
                 'drop_rate':0.1,
                 'qkv_bias':False}

torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
model.eval()


start_context="Every effort moves you"
tokenizer=tiktoken.get_encoding('gpt2')
token_ids=generate_text_simple(model=model, idx=text_to_token_ids(start_context, tokenizer),
                               max_new_tokens=10, context_size=GPT_CONFIG_124M['context_length'])
print(f"Output text\n", token_ids_to_text(token_ids, tokenizer))

Output text
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


In [9]:
inputs=torch.tensor([[16833, 3626, 6100], # every effort moves
                     [40, 1107, 588]]) # I really like
targets=torch.tensor([[3626, 6100, 345], # effort moves you
                      [1107, 588, 11311]]) # really like chocolate
with torch.no_grad(): logits=model(inputs)
probs=torch.softmax(logits, dim=-1) # prbability of each token in vocabulary
print(f"{probs.shape=}")

token_ids=torch.argmax(probs, dim=-1, keepdim=True)
print(f"{token_ids.shape=}")
print(f"Target batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Output batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

probs.shape=torch.Size([2, 3, 50257])
token_ids.shape=torch.Size([2, 3, 1])
Target batch 1:  effort moves you
Output batch 1:  Armed heNetflix


In [12]:
# softmax probability score corresponding to the target tokens
text_idx=0
target_prob_1=probs[text_idx, [0,1,2], targets[text_idx]]
print(f"{target_prob_1=}")
text_idx=1
target_prob_2=probs[text_idx, [0,1,2], targets[text_idx]]
print(f"{target_prob_2=}")

target_prob_1=tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])
target_prob_2=tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


In [21]:
# unsqueeze(-1) to changes the index shape from (2,3) to (2,3,1), to tell PyTorch: 
# "For every (i,j) pair, look at this specific index in the third dimension."
# dim=2 specifies that the gathering happens across the probability values (the 50257-length vector).
out=torch.gather(probs, dim=2, index=targets.unsqueeze(-1)).squeeze(-1)
print(f"{out.shape=}\n{out}")

out.shape=torch.Size([2, 3])
tensor([[7.4540e-05, 3.1061e-05, 1.1563e-05],
        [1.0337e-05, 5.6776e-05, 4.7559e-06]])


In [25]:
target_prob=torch.cat((target_prob_1, target_prob_2))
print(f"{target_prob.shape=}")
log_probs=target_prob.log()
print(f"{log_probs=}")
avg_log_probs=torch.mean(log_probs)
print(f"{avg_log_probs=}")

target_prob.shape=torch.Size([6])
log_probs=tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])
avg_log_probs=tensor(-10.7940)


In [28]:
print(f"{logits.shape=}, {targets.shape=}")
# flatten the tensors by combining them over the batch dimension
logits_flat=logits.flatten(0,1) # unscaled model outputs before softmax
targets_flat=targets.flatten() # token indices we want LLM to generate
print(f"{logits_flat.shape=},{targets_flat.shape=}") 
loss=torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(f"{loss=}")

logits.shape=torch.Size([2, 3, 50257]), targets.shape=torch.Size([2, 3])
logits_flat.shape=torch.Size([6, 50257]),targets_flat.shape=torch.Size([6])
loss=tensor(10.7940)


In [30]:
filepath='../the-verdict.txt'
with open(filepath, 'r', encoding='utf-8') as f: text_data=f.read()
total_characters=len(text_data)
total_tokens=len(tokenizer.encode(text_data))
print(f"{total_characters=}, {total_tokens=}")

total_characters=20479, total_tokens=5145


In [34]:
torch.manual_seed(123)

train_ratio=0.9
split_idx=int(train_ratio*len(text_data))
train_data=text_data[:split_idx]
val_data=text_data[split_idx:]
print(f"{split_idx=}, {len(train_data)=}, {len(val_data)=}")
train_loader=create_dataloader_v1(train_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'],
                                  stride=GPT_CONFIG_124M['context_length'], drop_last=True, shuffle=True, num_workers=0)
val_loader=create_dataloader_v1(val_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'], 
                                stride=GPT_CONFIG_124M['context_length'], drop_last=False, shuffle=False, num_workers=0)
print(f"{len(train_loader)=}, {len(val_loader)=}")
print("\nTrain loader:")
for x, y in train_loader: print(x.shape, y.shape)
print("Val loader:")
for x, y in val_loader: print(x.shape, y.shape)

split_idx=18431, len(train_data)=18431, len(val_data)=2048
len(train_loader)=9, len(val_loader)=1

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
Val loader:
torch.Size([2, 256]) torch.Size([2, 256])


In [43]:
if any(x in platform for x in ('linux','win32')): 
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
elif platform=='darwin':  
    device=torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f"{device=}")
model.to(device)
with torch.no_grad():
    train_loss=calc_loss_loader(train_loader, model, device)
    val_loss=calc_loss_loader(val_loader, model, device)
print(f"{train_loss=}, {val_loss=}")

device=device(type='mps')
train_loss=10.987583054436577, val_loss=10.98110580444336


torch.int64

In [ ]:
5.3